Here I am checking if the models trained by AttackER paper authors' used all the same hyperparameters.

In [4]:

from huggingface_hub import list_models, HfApi

# Method 2: Using HfApi client (useful if you need a custom token/endpoint)
api = HfApi()  # optional: use if private models need auth
models = api.list_models(author="Cyber-ThreaD")

for model in models:
    print(model.modelId, model.downloads, model.tags)

Cyber-ThreaD/SecureBERT-APTNER 4 ['transformers', 'tensorboard', 'safetensors', 'roberta', 'token-classification', 'generated_from_trainer', 'base_model:ehsanaghaei/SecureBERT', 'base_model:finetune:ehsanaghaei/SecureBERT', 'license:bigscience-openrail-m', 'endpoints_compatible', 'region:us']
Cyber-ThreaD/SecureBERT-DNRTI 3 ['transformers', 'tensorboard', 'safetensors', 'roberta', 'token-classification', 'generated_from_trainer', 'base_model:ehsanaghaei/SecureBERT', 'base_model:finetune:ehsanaghaei/SecureBERT', 'license:bigscience-openrail-m', 'endpoints_compatible', 'region:us']
Cyber-ThreaD/SecureBERT-CyNER 2058 ['transformers', 'tensorboard', 'safetensors', 'roberta', 'token-classification', 'generated_from_trainer', 'base_model:ehsanaghaei/SecureBERT', 'base_model:finetune:ehsanaghaei/SecureBERT', 'license:bigscience-openrail-m', 'endpoints_compatible', 'region:us']
Cyber-ThreaD/CyBERT-CyNER 5 ['transformers', 'tensorboard', 'safetensors', 'roberta', 'token-classification', 'genera

In [6]:
import re
import pandas as pd
from huggingface_hub import ModelCard, HfApi

AUTHOR = "Cyber-ThreaD"
api = HfApi()

# Grab all model ids for the account
model_ids = [m.modelId for m in api.list_models(author=AUTHOR)]

# The hyperparameter block looks like:
#   learning_rate: 2e-05
#   train_batch_size: 2
#   ...
# (sometimes rendered as markdown list items, i.e. prefixed with "- ")
HEADER_RE = re.compile(
    r"The following hyperparameters were used during training:(.*?)(?:\n#|\Z)",
    re.DOTALL | re.IGNORECASE,
)
KV_RE = re.compile(r"^\s*[-*]?\s*([A-Za-z0-9_]+):\s*(.+?)\s*$", re.MULTILINE)


def extract_hparams(card_text: str) -> dict:
    m = HEADER_RE.search(card_text)
    if not m:
        return {}
    block = m.group(1)
    return {k: v for k, v in KV_RE.findall(block)}


rows = {}
for mid in model_ids:
    try:
        card = ModelCard.load(mid)
        rows[mid] = extract_hparams(card.text)
    except Exception as e:
        rows[mid] = {"_error": str(e)}

df = pd.DataFrame(rows).T.sort_index()
df


,learning_rate,train_batch_size,eval_batch_size,seed,optimizer,lr_scheduler_type,num_epochs
Cyber-ThreaD/CyBERT-APTNER,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/CyBERT-AttackER,2e-05,2,2,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/CyBERT-CyNER,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/CyBERT-DNRTI,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/DeBERTa-APTNER,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/DeBERTa-CyNER,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/DeBERTa-DNRTI,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/DeBERTa-v3-AttackER,2e-05,2,2,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/RoBERTa-APTNER,2e-05,8,8,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0
Cyber-ThreaD/RoBERTa-AttackER,2e-05,2,2,42,"Adam with betas=(0.9,0.999) and epsilon=1e-08",linear,10.0


In [ ]:
# Are the hyperparameters the same across all models?
summary = pd.DataFrame({
    "unique_values": df.nunique(dropna=True),
    "values": [sorted(df[c].dropna().unique().tolist()) for c in df.columns],
})
summary["same_for_all"] = summary["unique_values"] <= 1
print("Constant across all models:")
print(summary[summary["same_for_all"]]["values"].to_string(), "\n")
print("Differs between models:")
print(summary[~summary["same_for_all"]][["unique_values", "values"]].to_string())
